In [1]:
import sys
sys.path.insert(0, "/home/ci2dt2-ai/Proyectos/Psiquiatria")
import kd_common as kdc
import pandas as pd
import numpy as np

cfg = kdc.get_config("ansiedad")
kdc.set_global_seed(kdc.SEED)
print(cfg)

DisorderConfig(name='ansiedad', pipeline_feat_path=PosixPath('/home/ci2dt2-ai/Proyectos/Psiquiatria/Ansiedad/MachineLearning/pipeline_feat_ansiedad.py'), librosa_features_csv=PosixPath('/home/ci2dt2-ai/Proyectos/Psiquiatria/Ansiedad/MachineLearning/features_ansiedad_5.csv'), egemaps_features_csv=PosixPath('/home/ci2dt2-ai/Proyectos/Psiquiatria/Ansiedad/MachineLearning/features_ansiedad_egemaps.csv'), experiment_dir=PosixPath('/home/ci2dt2-ai/Proyectos/Psiquiatria/Ansiedad/KD_BiLSTM_XGBoost'))


# 01 — Dataset maestro y Teacher BiLSTM

Fases 1-5 del protocolo. Teacher = BiLSTM sobre secuencias de embeddings preentrenados
(wav2vec2-large-robust) a nivel PACIENTE (arquitectura ya validada en
`dl_script_bilstm_final.py`, ROC-AUC~0.77 sobre 50 repeticiones). Se ensambla sobre 10
particiones de 5 folds distintas para estabilizar el OOF (un solo split sobre 79
pacientes es muy ruidoso: verificado empíricamente entre 0.33 y 0.89 de AUC por fold
según qué pacientes caen juntos). La probabilidad por paciente se propaga a cada
segmento de ese paciente para el Student (notebook 02).

In [2]:
master = kdc.build_master_table(cfg)
split = kdc.split_dev_test(master)
folds = kdc.make_folds(master, split)

master.to_csv(cfg.experiment_dir / "data" / "master_table.csv", index=False)
split.to_csv(cfg.experiment_dir / "data" / "development_test_split.csv", index=False)
folds.to_csv(cfg.experiment_dir / "data" / "folds_5cv.csv", index=False)

dev = kdc.build_dev_frame(master, split, folds)
test = kdc.build_test_frame(master, split)
print(f"dev: {len(dev)} muestras, {dev.patient_id.nunique()} pacientes")
print(f"test: {len(test)} muestras, {test.patient_id.nunique()} pacientes")
assert set(dev.patient_id) & set(test.patient_id) == set(), "fuga paciente dev/test"
dev.groupby("fold").agg(n_samples=("sample_id","size"), n_patients=("patient_id","nunique"), pos_rate=("label","mean"))

Audios encontrados : 79
  sin ansiedad (0): 60
  con ansiedad (1): 19


dev: 4238 muestras, 63 pacientes
test: 986 muestras, 16 pacientes


,n_samples,n_patients,pos_rate
fold,,,
1,847,12,0.270366
2,849,13,0.256773
3,853,13,0.261430
4,857,13,0.264877
5,832,12,0.275240


## Fase 4 — Teacher OOF (embeddings wav2vec2, ensamblado)

In [3]:
patient_oof = kdc.train_patient_teacher_oof_ensemble(
    "ansiedad", dev, cfg.experiment_dir / "teacher",
)
oof = kdc.broadcast_patient_scores(
    dev[["sample_id", "patient_id", "fold", "label"]].rename(columns={"label": "y_real"}),
    patient_oof, "p_teacher_OOF",
)
oof = oof[["sample_id", "patient_id", "fold", "y_real", "p_teacher_OOF"]]
oof.to_csv(cfg.experiment_dir / "teacher" / "teacher_predictions_development_oof.csv", index=False)
patient_oof.to_csv(cfg.experiment_dir / "teacher" / "teacher_patient_oof.csv", index=False)
print(f"OOF: {len(oof)} muestras (segmento), {len(patient_oof)} pacientes")
oof.head()

19:53:14 [INFO]    Cargados: 79 pacientes | shape=(79, 157, 1024) | clase0=60 | clase1=19


OOF: 4238 muestras (segmento), 63 pacientes


,sample_id,patient_id,fold,y_real,p_teacher_OOF
0,001_recortado_denoised__seg0,001,1,1,0.288782
1,001_recortado_denoised__seg1,001,1,1,0.288782
2,001_recortado_denoised__seg10,001,1,1,0.288782
3,001_recortado_denoised__seg11,001,1,1,0.288782
4,001_recortado_denoised__seg12,001,1,1,0.288782


In [4]:
# Sanity check OOF (Anexo A del PDF): el fold que predijo cada muestra
# nunca debe coincidir con los folds usados para ENTRENAR ese Teacher, es
# decir, el fold asociado a cada muestra en `oof` debe ser el mismo que en
# `dev` (así sabemos qué modelo temporal la predijo) y por construcción ese
# modelo se entrenó excluyendo justamente ese fold.
check = oof.merge(dev[["sample_id", "fold"]], on="sample_id", suffixes=("_oof", "_dev"))
assert (check["fold_oof"] == check["fold_dev"]).all(), "Inconsistencia de fold en el OOF"
print("OK: cada muestra fue predicha por el Teacher que excluyó su propio fold.")

OK: cada muestra fue predicha por el Teacher que excluyó su propio fold.


## Fase 5 — Métricas del Teacher por fold (a nivel paciente)

Se calculan sobre `patient_oof` (una fila por paciente), no sobre el `oof` propagado a
segmentos — de lo contrario los pacientes con más segmentos pesarían más en la métrica.
Umbral=0.5 es solo diagnóstico; la selección real de umbral (Paso 13) ocurre en el
notebook 02 sobre las predicciones del Student.

In [5]:
rows = []
for fold in sorted(patient_oof["fold"].unique()):
    sub = patient_oof[patient_oof["fold"] == fold]
    m = kdc.compute_binary_metrics(sub["y_real"].to_numpy(), sub["p_teacher_OOF"].to_numpy(), threshold=0.5)
    m["fold"] = fold
    m["n_patients"] = len(sub)
    rows.append(m)
teacher_metrics_cv = pd.DataFrame(rows)[["fold", "n_patients", "roc_auc", "pr_auc", "f1", "precision", "recall", "specificity"]]
teacher_metrics_cv.to_csv(cfg.experiment_dir / "teacher" / "teacher_metrics_cv.csv", index=False)
print(teacher_metrics_cv.to_string(index=False))
print()
print(f"ROC-AUC medio (paciente, por fold): {teacher_metrics_cv['roc_auc'].mean():.3f} +/- {teacher_metrics_cv['roc_auc'].std():.3f}")
from sklearn.metrics import roc_auc_score
print(f"ROC-AUC pooled (todos los pacientes OOF juntos): {roc_auc_score(patient_oof.y_real, patient_oof.p_teacher_OOF):.3f}")

 fold  n_patients  roc_auc   pr_auc  f1  precision  recall  specificity
    1          12 0.814815 0.555556 0.0        0.0     0.0          1.0
    2          13 0.566667 0.361111 0.0        0.0     0.0          1.0
    3          13 0.533333 0.376923 0.0        0.0     0.0          1.0
    4          13 0.433333 0.261111 0.0        0.0     0.0          0.9
    5          12 0.592593 0.347222 0.0        0.0     0.0          1.0

ROC-AUC medio (paciente, por fold): 0.588 +/- 0.140
ROC-AUC pooled (todos los pacientes OOF juntos): 0.593
